### Chapter 10: Tool Engineering
### Topic 1: What a Tool Is to an LLM


### End-to-End Flow of Tool Calling Fundamentals

This chapter explains how an LLM interacts with external systems using **Tool Calling**. The most important idea is that **an LLM never executes code**. It only requests that the application execute a tool on its behalf.

The application remains in complete control of what actually happens.

The program performs the following steps in order:

1. User asks a question.
2. Application sends the system prompt, conversation, and available tool descriptions to the LLM.
3. LLM decides whether it needs a tool.
4. If required, the LLM generates a tool call.
5. The application validates the tool request.
6. The application executes the real Python function.
7. The application sends the tool result back to the LLM.
8. The LLM generates the final answer.

---

### Why Does Tool Calling Exist?

Suppose a customer asks:

> **"What is the balance of my FD reference FD123456?"**

Can the LLM answer this?

No.

Why?

Because the LLM does not know:

- Your customer database
- Your Qdrant collection
- Your CSV files
- Your REST APIs
- Your internal systems

The LLM only knows what was learned during training and what is present in the current conversation.

If the required information is outside the conversation, the LLM cannot access it by itself.

This is why **Tool Calling** exists.

---

### What Can an LLM Actually Do?

Many people think an LLM can execute Python code.

It cannot.

An LLM can only:

- Read text
- Understand text
- Generate text

It cannot:

- Execute Python
- Query SQL
- Search Qdrant
- Read a CSV
- Call an API
- Access the internet
- Send emails

Every real-world action must be performed by the application.

This is the most important concept in Tool Calling.

---

### What Is a Tool?

A tool is simply a capability provided by the application.

Usually it is a Python function.

Example:

```python
validate_fd_reference()

search_knowledge_base()

send_email()

book_meeting()
```

These are ordinary Python functions written by the developer.

The LLM never owns them.

The application owns them.

---

### How Does the LLM Know a Tool Exists?

The LLM cannot discover Python functions automatically.

The application must tell it which tools are available.

Before every request, the application sends:

- System Prompt
- Conversation History
- Tool Descriptions

The LLM reads all of this before generating a response.

---

### What Is a Tool Schema?

The LLM cannot read Python code.

Instead, it receives a **tool schema**.

Example:

```text
Tool Name

search_knowledge_base

Description

Search the banking knowledge base.

Input

query : string
```

Notice something important.

The LLM never sees:

```python
def search_knowledge_base():
```

It only sees the tool description.

The schema is the only interface between the LLM and the application.

---

### Why Is the Tool Description Important?

The description tells the LLM:

- What the tool does.
- When it should be used.
- What inputs are required.

Example:

```text
Use this tool whenever the user asks a question about banking policies.
```

If the description is poor, the LLM may:

- Never call the tool.
- Call the wrong tool.
- Call it at the wrong time.

Good tool descriptions significantly improve tool selection.

---

### What Is the Role of the System Prompt?

The tool description explains **what a tool does**.

The system prompt explains **how the model should behave**.

Example:

```text
If an FD reference number is present,
you must call validate_fd_reference()
before answering.
```

The LLM uses both pieces of information while making its decision.

---

### How Does the LLM Decide to Use a Tool?

After reading:

- System Prompt
- Conversation
- Tool Descriptions

the LLM starts reasoning.

Suppose the user asks:

> "Check FD123456."

The model thinks:

- I don't know this FD.
- There is a tool called validate_fd_reference().
- The system prompt tells me to use it.
- I should request that tool.

Notice that this is only a reasoning step.

Nothing has been executed yet.

---

### What Does the LLM Actually Generate?

The LLM does not execute Python.

Instead, it generates a structured request.

Example:

```text
Tool Name

validate_fd_reference

Arguments

reference_number = FD123456
```

This is called a **tool call**.

Think of it as saying:

> "Please execute this tool for me."

Nothing has happened yet.

---

### Who Executes the Tool?

The application receives the tool request.

Example:

```text
Tool

validate_fd_reference

Reference Number

FD123456
```

Now the application decides what to do.

It can:

- Execute the tool.
- Reject the request.
- Validate the inputs.
- Check user permissions.

The application has complete control.

The LLM has none.

---

### What Is the Dispatch Layer?

The application must determine which Python function should run.

Example:

```python
if tool_name == "validate_fd_reference":
    validate_fd_reference(...)

elif tool_name == "search_knowledge_base":
    search_knowledge_base(...)
```

This decision-making code is called the **dispatch layer**.

It connects the LLM's request to the correct Python function.

---

### Why Validate Inputs?

Never trust the model's arguments.

Suppose the LLM generates:

```text
reference_number = ""
```

or

```text
reference_number = FDABCXYZ
```

The application must validate:

- Is the argument present?
- Is the format correct?
- Does the user have permission?

Only after validation should the tool execute.

---

### What Happens After the Tool Executes?

Suppose the Python function returns:

```text
Customer

Sandipan Paul

FD Amount

₹5,00,000

Status

Active
```

The application sends this information back to the LLM.

This is called the **tool result**.

Now the LLM finally has the required information.

---

### How Does the LLM Generate the Final Answer?

The LLM receives:

```text
Tool Result

Customer

Sandipan Paul

FD Amount

₹5,00,000
```

Now it continues generating.

Example:

```text
Your FD reference FD123456 is active.

Current Balance

₹5,00,000.
```

The answer is based on the tool result, not on the model's memory.

---

### Can the LLM Use Multiple Tools?

Yes.

Suppose the customer asks:

> "Check my FD and email me the details."

The LLM may decide:

```text
Tool 1

validate_fd_reference()

------------------------

Tool 2

send_email()
```

The application executes each tool separately.

After receiving all results, the LLM generates one final response.

---

### Complete Execution Flow

```text
Customer

What is my FD balance?

------------------------

Application

System Prompt

Conversation

Tool Schemas

------------------------

LLM

Reads Everything

------------------------

LLM

Requests

validate_fd_reference()

------------------------

Application

Validates Request

------------------------

Application

Executes Python Function

------------------------

Application

Returns Tool Result

------------------------

LLM

Generates Final Answer
```

This entire sequence is called **Tool Calling**.

---

### Why Is This Architecture Used?

Imagine the LLM could execute Python directly.

A malicious user might ask:

> "Delete every customer."

If the LLM had unrestricted execution rights, it could perform dangerous operations.

Instead, the architecture separates responsibilities.

**LLM**

- Understands language.
- Makes decisions.
- Requests tools.

**Application**

- Executes code.
- Reads databases.
- Calls APIs.
- Validates requests.
- Applies security rules.
- Logs every operation.

This separation makes AI systems secure, auditable, and reliable.

---

### How Is It Implemented in This Project?

Our project already contains tools like:

```python
validate_fd_reference()

search_knowledge_base()

finalize_answer_with_citations()
```

The LLM only sees their schemas.

When it decides to use one:

1. It generates a tool call.
2. The dispatch layer identifies the correct Python function.
3. The function executes.
4. The result is returned.
5. The LLM generates the final answer.

This same pattern is used throughout the project.

---

### Production Engineer's Perspective

Always remember these rules.

- The LLM never executes code.
- The LLM never accesses databases.
- The LLM never calls APIs.
- The LLM only requests tools.
- The application validates every request.
- The application executes every tool.
- The application returns the result.
- The LLM generates the final answer.

A useful mental model is:

```text
LLM

Smart Decision Maker

------------------------

Application

Real Worker
```

The LLM decides **what information it needs**.

The application performs the actual work.

This separation is the foundation of every production-grade AI agent.

---

### Why Is Validation Required?

The LLM generates the tool arguments.

Those arguments should **never** be trusted automatically.

Suppose the model generates:

```text
reference_number = ""
```

or

```text
reference_number = ABCXYZ
```

or

```text
customer_id = -100
```

The application must validate:

- Required fields are present.
- Data types are correct.
- Values are within allowed limits.
- User has permission.

Only after validation should the tool execute.

Never assume the LLM always generates valid inputs.

---

### Why Is Security Important?

The LLM can request a tool.

It should never control what actually happens.

Suppose a customer asks:

> Delete every customer record.

The LLM might request:

```text
delete_customer_records()
```

The application should reject this request because the customer is not authorized.

The application—not the LLM—is responsible for:

- Authentication
- Authorization
- Permission checks
- Business rules
- Access control

The LLM can request.

The application decides.

---

### Why Is Logging Important?

Every tool call should be logged.

Example:

```text
Timestamp

10:15:24

------------------------

Tool

search_knowledge_base

------------------------

Arguments

premature FD withdrawal

------------------------

Execution Time

42 ms

------------------------

Status

Success
```

Logs help with:

- Debugging
- Monitoring
- Auditing
- Compliance
- Performance analysis

Without logs, production issues become difficult to investigate.

---

### Why Monitor Tool Calls?

Production systems monitor every tool.

Typical metrics include:

- Number of tool calls
- Success rate
- Failure rate
- Average latency
- Peak latency
- Timeout count

Example:

```text
search_knowledge_base

Calls

15,000

Average Latency

45 ms

Failure Rate

0.2%
```

These metrics help identify performance problems before customers notice them.

---

### Why Does Tool Calling Increase Latency?

Every tool call adds another interaction between the LLM and the application.

Without tools:

```text
User

LLM

Answer
```

One model call.

With one tool:

```text
User

LLM

Application

LLM

Answer
```

Two model calls.

With two tools:

```text
User

LLM

Application

LLM

Application

LLM

Answer
```

Three model calls.

Each additional tool increases:

- Latency
- API cost
- Overall response time

---

### Why Do Agent Loops Have max_steps?

Suppose the model repeatedly calls tools.

Example:

```text
Tool A

Tool B

Tool A

Tool B

Tool A

...
```

This could continue forever.

To prevent infinite loops, agent frameworks define a maximum number of tool calls.

Example:

```text
max_steps = 5
```

After five tool calls, the application stops the agent.

This prevents:

- Infinite execution
- High API costs
- Poor user experience

---

### How Do We Debug Tool Calling?

Failures can happen in different places.

#### Case 1: Tool Never Called

The user asks:

> Check FD123456.

The model answers directly without calling the validation tool.

Possible reasons:

- Weak system prompt.
- Poor tool description.
- Incorrect prompting.

---

#### Case 2: Wrong Tool Called

The model calls:

```text
search_knowledge_base()
```

instead of

```text
validate_fd_reference()
```

Possible reasons:

- Similar tool descriptions.
- Ambiguous instructions.

---

#### Case 3: Invalid Arguments

Example:

```text
reference_number = ""
```

The model selected the correct tool but generated incorrect inputs.

Possible reasons:

- Poor schema.
- Weak validation rules.
- Ambiguous user query.

---

#### Case 4: Tool Execution Failed

Example:

```text
Database Connection Failed
```

The LLM behaved correctly.

The Python function failed.

This is a software issue, not an AI issue.

---

#### Case 5: Tool Returned Incorrect Data

The database itself contains incorrect information.

Again, this is not an LLM problem.

Understanding where the failure occurred saves significant debugging time.

---

### How Should Tool Descriptions Be Written?

A tool description should explain:

- What the tool does.
- When it should be used.
- Required inputs.

Good Example:

```text
Search banking policy documents.

Use this tool whenever the user asks about FD, Loan, or Insurance policies.
```

Bad Example:

```text
Uses FAISS index with cosine similarity and HNSW search.
```

The implementation details are useful for developers, not for the LLM.

The LLM only needs enough information to decide **when** to call the tool.

---

### What Belongs in the System Prompt?

The tool description explains the tool itself.

The system prompt explains the overall behavior.

Example:

```text
If an FD reference number is present,
always validate it before answering.
```

This instruction affects the model's reasoning across the entire conversation.

---

### Where Should Validation Happen?

There are two common approaches.

#### Validation in Dispatch Layer

The dispatcher validates inputs before calling the function.

Advantages:

- Prevents invalid function calls.
- Centralized validation.

---

#### Validation Inside the Tool

Each Python function validates its own inputs.

Advantages:

- Tool becomes self-contained.
- Can be reused elsewhere.

---

#### Production Recommendation

Use both.

The dispatcher performs basic validation.

Each tool performs its own business validation.

This provides defense in depth.

---

### Alternatives

#### Structured Tool Calling

The LLM requests external capabilities.

Best choice when interacting with:

- Databases
- APIs
- Vector Databases
- Enterprise Systems

This is the industry-standard approach.

---

#### No Tool Calling

The LLM answers using only:

- Training knowledge
- Conversation history

Suitable only when external information is unnecessary.

---

#### Direct Code Execution

The LLM executes Python directly.

Advantages:

- Very flexible.

Disadvantages:

- Major security risk.
- Difficult to audit.
- Difficult to restrict.
- Not recommended for enterprise systems.

---

### Common Production Mistakes

#### Mistake 1

Assuming the LLM executes Python.

Reality:

The application executes Python.

---

#### Mistake 2

Trusting model-generated arguments.

Reality:

Always validate.

---

#### Mistake 3

Poor tool descriptions.

Result:

Wrong tool selection.

---

#### Mistake 4

Not logging tool executions.

Result:

Very difficult debugging.

---

#### Mistake 5

Treating every failure as an AI problem.

Reality:

Failures may occur in:

- Prompt
- Tool description
- Dispatch layer
- Python code
- Database
- Network
- External API

Each requires a different debugging approach.

---

### Production Engineer's Perspective

A production Tool Calling system is much more than connecting an LLM to Python.

It must provide:

- Input validation
- Authentication
- Authorization
- Secure execution
- Logging
- Monitoring
- Error handling
- Retry mechanisms
- Cost control
- Latency control
- Loop prevention
- Auditing
- Debugging support

The LLM is responsible for **reasoning**.

The application is responsible for **execution**.

Keeping these responsibilities separate is what makes Tool Calling secure, scalable, and production-ready.

---

### Lead-Level Interview Questions

**Basic**

- Q: Can a language model actually execute a tool itself?  
  A: No — a model can only generate text or structured output. When it "calls a tool," it's producing a structured request (a `tool_use` block naming the tool and its arguments), and a completely separate piece of application code decides whether and how to actually execute the corresponding real function. The model never runs anything itself.

- Q: What information does a tool's schema actually provide, and to whom?  
  A: It provides the model with everything it needs to decide when and how to request the tool — a name, a natural-language description of its purpose and appropriate usage, and the structure of expected input arguments. It does not provide, and the model never sees, the tool's actual underlying implementation code.

**Intermediate**

- Q: Walk through what happens, step by step, from a model deciding to call `validate_fd_reference` to that tool's result influencing the model's final answer.  
  A: The model, based on the conversation and the tool's description, generates a `tool_use` block naming `validate_fd_reference` with a reference number argument. The application's dispatch logic reads this block, matches the name, and calls the real `validate_fd_reference()` Python function with that argument — this function performs an actual database lookup, entirely outside the model. The result is packaged as a `tool_result` block and sent back to the model as a new conversation turn. Only now does the model see what happened, and it continues generating, incorporating that result into its response.

- Q: Why is it important that the application code, not the model, decides whether to actually honor a tool-call request?  
  A: This separation is what makes every safeguard in this project's architecture possible — argument validation, access control (like Chapter 9 Topic 4's exact-match-only rule for customer records), audit logging, and error handling all depend on there being a real, controllable checkpoint between the model's request and any consequential action. If the model could execute code directly, none of these safeguards could exist.

**Advanced**

- Q: A teammate says "the model looked up the customer's FD record." What's imprecise about this description, and why does the distinction matter practically?  
  A: The model didn't look anything up — it requested a lookup, and the application's real code performed it. This distinction matters practically because it clarifies where a bug actually lives: if the wrong record was returned, the bug is in the real lookup function or the data it queries, not in the model's "reasoning" — and conversely, if the model failed to request a lookup it should have, that's a model-behavior or prompt-description problem, not a bug in the lookup function at all. Conflating these leads to debugging effort aimed at the wrong part of the system.

- Q: Design the division of responsibility between a tool's schema description and the system prompt for a case where a tool should be called under a specific, narrow condition.  
  A: The tool's own description should explain what the tool does and give the model enough context to recognize relevant situations on its own — but for a hard requirement ("you MUST call this specific tool whenever X specific pattern appears"), that rule is more reliably enforced as an explicit instruction in the system prompt, which governs the model's behavior across the whole interaction, rather than relying solely on the tool's own description to convey a mandatory behavioral rule. This mirrors exactly how the project's existing system prompt explicitly instructs mandatory `validate_fd_reference` calls, rather than leaving that requirement to the tool's description alone.

**Scenario-based**

- Q: In production, a tool call's arguments occasionally contain unexpected or malformed data that causes the underlying function to error. How do you think about where the fix belongs?  
  A: First determine whether the model is generating malformed arguments because the tool schema's expected input format wasnns't clearly specified (a schema-design problem, this chapter's next topic), or whether the arguments are well-formed but the underlying function doesn't handle a legitimate edge case in real data (an ordinary software bug in the real implementation, unrelated to the model at all). These require entirely different fixes — improving the schema's clarity versus fixing the function's own logic — and correctly attributing the failure to the right side of the model/application boundary is the first step before attempting either fix.

**Follow-up questions to expect:**

- "What would you log to distinguish a model tool-selection problem from an underlying function bug?"
- "How does this model-cannot-execute-code principle relate to prompt injection risk?"


### Module 1: Schema (Documentation for the Model) vs Real Function (Documentation for Nobody)

Build a tool exactly like the project's existing `validate_fd_reference` -- a schema dict the model reads, and a completely separate real function the model never sees.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Schema (for the model) vs real function (the model never
# sees this code at all -- it is pure application-side logic)
# ------------------------------------------------------------------

# the SCHEMA -- this is the ONLY thing the model ever sees about this
# tool. It is pure documentation, read by the model to decide when
# and how to request a call. It contains NO actual logic.
VALIDATE_FD_REFERENCE_SCHEMA = {
    "name": "validate_fd_reference",
    "description": (
        "Look up an FD account by its reference number to confirm it exists "
        "and check its current status. Call this whenever an email contains "
        "something that looks like an FD reference number."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "reference_number": {"type": "string", "description": "The FD reference number to validate."},
        },
        "required": ["reference_number"],
    },
}

# the REAL FUNCTION -- actual logic, entirely separate from the schema
# above, and NEVER seen or executed by the model itself
FD_RECORDS = {
    "BJ2019FD7717": {"customer_name": "Shobha Chopra", "amount": 391000, "status": "Closed (Premature)"},
    "BJ2022FD6918": {"customer_name": "Anita Mishra", "amount": 160000, "status": "Active"},
}

def validate_fd_reference(reference_number: str) -> dict:
    """The REAL implementation. This code is never sent to the model,
    never read by the model, and never executed by the model -- only
    the application's own dispatch layer ever calls this directly."""
    record = FD_RECORDS.get(reference_number.strip())
    if record:
        return {"found": True, **record}
    return {"found": False}


print("=" * 70)
print("MODULE 1: SCHEMA (model-facing) vs REAL FUNCTION (application-only)")
print("=" * 70)
print("What the model sees (the schema):")
schema_name = VALIDATE_FD_REFERENCE_SCHEMA["name"]
schema_desc = VALIDATE_FD_REFERENCE_SCHEMA["description"][:70]
print(f"  name: {schema_name}")
print(f"  description: {schema_desc}...")
print("  (NO implementation logic is present in this schema at all)")

print("\nWhat the model NEVER sees (the real function):")
direct_result = validate_fd_reference("BJ2019FD7717")
print(f"  validate_fd_reference('BJ2019FD7717') -> {direct_result}")
print("  (This is real Python code with a real dictionary lookup --")
print("   completely invisible to the model, which only ever sees")
print("   the schema above and, later, this function's RETURNED result.)")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: SCHEMA (model-facing) vs REAL FUNCTION (application-only)
What the model sees (the schema):
  name: validate_fd_reference
  description: Look up an FD account by its reference number to confirm it exists and...
  (NO implementation logic is present in this schema at all)

What the model NEVER sees (the real function):
  validate_fd_reference('BJ2019FD7717') -> {'found': True, 'customer_name': 'Shobha Chopra', 'amount': 391000, 'status': 'Closed (Premature)'}
  (This is real Python code with a real dictionary lookup --
   completely invisible to the model, which only ever sees
   the schema above and, later, this function's RETURNED result.)

Module 1 complete. Run Module 2 next.


### Module 2: The Model Requests, the Dispatch Layer Decides

Simulate a model's tool_use request (since we can't call the real API offline), and run the REAL dispatch logic that reads that request and decides whether to actually execute anything.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Model REQUESTS a tool call; dispatch layer DECIDES and ACTS
#
# The simulated "model response" below stands in for a real API call.
# Everything AFTER that -- the dispatch, validation, and execution --
# is REAL, testable code implementing the actual separation this
# topic is about.
# ------------------------------------------------------------------

def simulate_model_tool_request(email_text: str) -> dict:
    """SIMULATES what a real model call would return as a tool_use
    block. Standing in for client.messages.create() -- the model has
    ALREADY decided to request this tool; nothing has executed yet."""
    import re
    match = re.search(r'\b([A-Z]{2}\d{4}FD\d{4})\b', email_text)
    if match:
        return {"tool_use": True, "name": "validate_fd_reference",
                 "input": {"reference_number": match.group(1)}}
    return {"tool_use": False}


def dispatch_tool_call(model_request: dict) -> dict:
    """The REAL application-side dispatch layer. This is where the
    separation this topic describes actually lives in code: it reads
    the model's REQUEST, validates it, and ONLY THEN decides to
    actually call the real function."""
    if not model_request.get("tool_use"):
        return {"status": "no_tool_requested"}

    tool_name = model_request["name"]
    tool_input = model_request["input"]

    # REAL validation happens HERE, before any real execution --
    # this is the checkpoint that only exists because the model
    # never executes anything directly itself
    if tool_name == "validate_fd_reference":
        ref = tool_input.get("reference_number", "")
        if not re.match(r'^[A-Z]{2}\d{4}FD\d{4}$', ref):
            return {"status": "validation_failed",
                    "reason": f"'{ref}' does not match expected reference number format -- refusing to execute."}
        # validation passed -- ONLY NOW does the real function actually run
        result = validate_fd_reference(ref)
        return {"status": "executed", "tool_name": tool_name, "result": result}

    return {"status": "unknown_tool"}


import re

TEST_EMAILS = [
    "Please check my FD BJ2019FD7717 status.",
    "Just checking in, no specific account mentioned.",
]

print("=" * 70)
print("MODULE 2: REQUEST -> DISPATCH -> VALIDATE -> EXECUTE (real separation)")
print("=" * 70)

for email in TEST_EMAILS:
    print(f"\nEmail: '{email}'")
    model_request = simulate_model_tool_request(email)
    print(f"  Model's REQUEST (nothing executed yet): {model_request}")

    dispatch_result = dispatch_tool_call(model_request)
    dispatch_status = dispatch_result["status"]
    print(f"  Dispatch layer's decision: {dispatch_status}")
    if dispatch_result["status"] == "executed":
        exec_result = dispatch_result["result"]
        print(f"    Real function actually ran. Result: {exec_result}")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: REQUEST -> DISPATCH -> VALIDATE -> EXECUTE (real separation)

Email: 'Please check my FD BJ2019FD7717 status.'
  Model's REQUEST (nothing executed yet): {'tool_use': True, 'name': 'validate_fd_reference', 'input': {'reference_number': 'BJ2019FD7717'}}
  Dispatch layer's decision: executed
    Real function actually ran. Result: {'found': True, 'customer_name': 'Shobha Chopra', 'amount': 391000, 'status': 'Closed (Premature)'}

Email: 'Just checking in, no specific account mentioned.'
  Model's REQUEST (nothing executed yet): {'tool_use': False}
  Dispatch layer's decision: no_tool_requested

Module 2 complete. Run Module 3 next.


### Module 3: What Happens When the Model's Request Is Malformed

Demonstrates concretely why the validation checkpoint matters -- a model requesting an invalid or malformed reference number gets caught by the dispatch layer BEFORE any real function call happens.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Malformed model request -- caught BEFORE real execution
#
# This demonstrates the actual, concrete VALUE of the separation --
# a malformed or unexpected request from the model never reaches the
# real function unvalidated, because dispatch is a real, controllable
# checkpoint the model cannot bypass.
# ------------------------------------------------------------------

# simulate a model request with a MALFORMED reference number -- this
# could happen from a typo in the email, a model error, or (in
# principle) an adversarial attempt to pass unexpected input
MALFORMED_REQUEST = {"tool_use": True, "name": "validate_fd_reference",
                       "input": {"reference_number": "'; DROP TABLE fd_records; --"}}

WELL_FORMED_REQUEST = {"tool_use": True, "name": "validate_fd_reference",
                        "input": {"reference_number": "BJ2022FD6918"}}

print("=" * 70)
print("MODULE 3: MALFORMED vs WELL-FORMED MODEL REQUESTS")
print("=" * 70)

print("\n[MALFORMED request -- e.g. from a model error or adversarial input]")
malformed_input = MALFORMED_REQUEST["input"]
print(f"  Model's request: {malformed_input}")
malformed_result = dispatch_tool_call(MALFORMED_REQUEST)
malformed_status = malformed_result["status"]
print(f"  Dispatch layer's decision: {malformed_status}")
if malformed_result["status"] == "validation_failed":
    print(f"  Reason: {malformed_result['reason']}")
    print("  The REAL function was NEVER CALLED with this input --")
    print("  the validation checkpoint caught it first, exactly because")
    print("  the model's request and the real execution are separate steps.")

print("\n[WELL-FORMED request -- a genuine, correctly-formatted reference number]")
wellformed_input = WELL_FORMED_REQUEST["input"]
print(f"  Model's request: {wellformed_input}")
wellformed_result = dispatch_tool_call(WELL_FORMED_REQUEST)
wellformed_status = wellformed_result["status"]
print(f"  Dispatch layer's decision: {wellformed_status}")
if wellformed_result["status"] == "executed":
    wellformed_res = wellformed_result["result"]
print(f"  Real function ran. Result: {wellformed_res}")

print("\nThis is the concrete, practical payoff of this topic's core claim:")
print("because the model can only ever REQUEST a tool call, never execute")
print("it directly, the application always has a real, controllable")
print("opportunity to validate before anything consequential happens --")
print("a malformed or unexpected request is caught here, not after the")
print("fact.")

print("\nModule 3 complete. All Chapter 10 Topic 1 modules done.")
print()
print("=" * 70)
print("CHAPTER 10 TOPIC 1 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  A model CANNOT execute code -- it can only generate a structured
  REQUEST (a tool_use block: name + arguments). Nothing has happened
  in the world yet at that point.

  The application's dispatch layer reads this request and decides
  whether and how to actually call the REAL function -- this is a
  real, controllable checkpoint the model cannot bypass.

  This separation is what makes validation, access control, audit
  logging, and error handling all possible -- demonstrated concretely
  in Module 3, where a malformed request was caught BEFORE any real
  execution happened.

  The tool's SCHEMA (name, description, input_schema) is documentation
  for the model. The tool's REAL FUNCTION is documentation for nobody --
  the model never sees or executes it.
""")


MODULE 3: MALFORMED vs WELL-FORMED MODEL REQUESTS

[MALFORMED request -- e.g. from a model error or adversarial input]
  Model's request: {'reference_number': "'; DROP TABLE fd_records; --"}
  Dispatch layer's decision: validation_failed
  Reason: ''; DROP TABLE fd_records; --' does not match expected reference number format -- refusing to execute.
  The REAL function was NEVER CALLED with this input --
  the validation checkpoint caught it first, exactly because
  the model's request and the real execution are separate steps.

[WELL-FORMED request -- a genuine, correctly-formatted reference number]
  Model's request: {'reference_number': 'BJ2022FD6918'}
  Dispatch layer's decision: executed
  Real function ran. Result: {'found': True, 'customer_name': 'Anita Mishra', 'amount': 160000, 'status': 'Active'}

This is the concrete, practical payoff of this topic's core claim:
because the model can only ever REQUEST a tool call, never execute
it directly, the application always has a real, 